Example of optimizing a strategy

We try to optimize a pre-defined strategy over a combination of 2 parameters

To do this, we have to write a generator function and a cost function.  The generator produces all the combinations of parameters you want to optimize.  The cost function will run the strategy for each parameter combination provided by the generator and return whatever metric you want to optimize, as well as any other metrics you want to see at the same time.

In this case, our cost metric will be the sharpe ratio of the strategy but we will also look at sortino and drawdowns at the same time. We also look at the number of trades generate and ignore results as unreliable if the number of trades is less than 10.

The optimizer uses multiple processes to run as fast as possible.  You can set the number of processes you want to use using the max_processes argument to the Optimizer constructor.  If you don't set this, the optimizer will the same number of processes as the CPU cores on your machine.

In this case, we are optimizing 2 parameters at the same time, but we can optimize 1 parameter or more than 2 as well.

In [ ]:
# %%checkall
import numpy as np
import polars as pl
import gambit as pq
from gambit.evaluator import compute_sharpe, compute_sortino, compute_maxdd_pct, compute_amean, compute_rolling_dd
from gambit.evaluator import compute_periods_per_year
from build_example_strategy import build_example_strategy


def generator():
    for stop_pct in [-0.002]:
        for ret_threshold in [0.002]:
            _ = (yield {'stop_pct': stop_pct, 'ret_threshold': ret_threshold})
            yield

def cost_func(suggestion):
    strategy = build_example_strategy(stop_pct=suggestion['stop_pct'], ret_threshold=suggestion['ret_threshold'])
    strategy.run()
    
    returns_df = strategy.df_returns()
    
    num_trades = len(strategy.df_trades())
    
    if num_trades < 10: return np.nan, {}
    
    returns = returns_df['ret'].to_numpy()
    equity = returns_df['equity'].to_numpy()
    dates = returns_df['timestamp'].to_numpy()
    
    periods_per_year = compute_periods_per_year(dates)
    
    amean = compute_amean(returns, periods_per_year)
    sharpe = compute_sharpe(returns, amean, periods_per_year)
    sortino = compute_sortino(returns, amean, periods_per_year)
    rolling_dd = compute_rolling_dd(dates, equity)
    maxdd = compute_maxdd_pct(rolling_dd[1])
    
    return sharpe, {'sortino': sortino, 'maxdd': maxdd, 'num_trades': num_trades}

optimizer = pq.Optimizer('example', generator(), cost_func, max_processes=1)
optimizer.run(raise_on_error=True)

In [ ]:
if pq.has_display():
    optimizer.plot_3d(x='stop_pct', y='ret_threshold', height=1500);

A stop_pct of -0.002 and a ret_threshold of around 0.002 seems to have a good sharpe and sortino and the number of trades which indicates statistical significance also has a hill.  The actual points you provided to the plot are shown with markers.

Lets look at the actual values of the sharpes, sortinos and drawdowns

In [ ]:
optimizer.df_experiments(sort_column='maxdd', ascending=False)

Lets look at the strategy at stop_pct of 0.003

In [ ]:
df = optimizer.df_experiments(sort_column='maxdd', ascending=False)
df.filter(pl.col('stop_pct') == -0.002)

Let's run the strategy at a stop_pct of -0.002 and ret_threshold of 0.002

In [ ]:
strategy = build_example_strategy(stop_pct=-0.002, ret_threshold=0.002)
strategy.run()
strategy.evaluate_returns(plot=pq.has_display());

This confirms that these sets of parameters generate a positive sharpe and sortino and lower drawdowns. Obviously, this is just a month of data and in production you would use a lot more data.